In [1]:
#import libraries
import yaml
import json
import re
from snowflake.snowpark.context import get_active_session

In [ ]:
# Get active Snowflake session
session = get_active_session()

# Explicitly set database and schema for this session
session.sql("USE DATABASE CUSTOMER_BILLING_DB").collect()
session.sql("USE SCHEMA PUBLIC").collect()

# Fully qualified Azure external stage
stage_name = "CUSTOMER_BILLING_DB.PUBLIC.AZURE_SNOW_STAGE"

# Create temporary file format in the current database/schema
session.sql("""
    CREATE OR REPLACE TEMP FILE FORMAT CUSTOMER_BILLING_DB.PUBLIC.CB_YAML_TEXT_FF
    TYPE = CSV
    FIELD_DELIMITER = NONE
    RECORD_DELIMITER = '\\n'
    SKIP_HEADER = 0
""").collect()

# Read dev_config.json from Azure
config_file_path = f"@{stage_name}/config/dev_config.json"

config_rows = session.sql(f"""
    SELECT $1
    FROM {config_file_path}
    (FILE_FORMAT => 'CUSTOMER_BILLING_DB.PUBLIC.CB_YAML_TEXT_FF')
""").collect()

if not config_rows:
    raise Exception("dev_config.json was not found or is empty.")

# Combine file lines and parse JSON
config_text = "\n".join(
    str(row[0]) for row in config_rows if row[0] is not None
)

config = json.loads(config_text)

# Read configuration values
database_name = config["database"]

yaml_folders = [
    config["control_yaml_path"],
    config["bronze_yaml_path"],
    config["silver_yaml_path"],
    config["gold_yaml_path"]
]

print("Configuration loaded successfully!")
print("Database:", database_name)
print("YAML folders:")

for folder in yaml_folders:
    print(folder)


In [ ]:
# Step 4: Discover YAML files dynamically from Azure stage

# Use the fully qualified stage name
stage_name = "CUSTOMER_BILLING_DB.PUBLIC.AZURE_SNOW_STAGE"

# Convert configured paths to fully qualified stage paths
yaml_folders = {
    "CONTROL": config["control_yaml_path"],
    "BRONZE": config["bronze_yaml_path"],
    "SILVER": config["silver_yaml_path"],
    "GOLD": config["gold_yaml_path"]
}

yaml_files_by_layer = {}

for layer, folder in yaml_folders.items():

    # Replace short stage name with fully qualified stage name
    folder = folder.replace(
        "@AZURE_SNOW_STAGE",
        f"@{stage_name}"
    )

    # List files from the configured Azure folder
    result = session.sql(f"LIST {folder}").collect()

    # Keep only YAML files
    files = [
        row["name"]
        for row in result
        if row["name"].lower().endswith((".yaml", ".yml"))
    ]

    yaml_files_by_layer[layer] = files

    print(f"\n{layer} YAML files:")
    for file in files:
        print(file)

print("\nYAML discovery completed.")

In [ ]:
# Create a named file format for multiline YAML files
session.sql("""
    CREATE OR REPLACE FILE FORMAT
    CUSTOMER_BILLING_DB.PUBLIC.CB_YAML_TEXT_FF
    TYPE = 'CSV'
    FIELD_DELIMITER = NONE
    RECORD_DELIMITER = '\\n'
    SKIP_HEADER = 0
""").collect()

stage_file = (
    "@CUSTOMER_BILLING_DB.PUBLIC.AZURE_SNOW_STAGE/"
    "config/yaml/control/etl_run_log.yaml"
)

# Read the YAML file as one record
rows = session.sql(f"""
    SELECT $1
    FROM {stage_file}
    (
        FILE_FORMAT => 'CUSTOMER_BILLING_DB.PUBLIC.CB_YAML_TEXT_FF'
    )
""").collect()

if not rows:
    raise Exception("Could not read etl_run_log.yaml")

yaml_text = "\n".join(
    str(row[0]) for row in rows if row[0] is not None
)

yaml_config = yaml.safe_load(yaml_text)

print("Total lines:", len(rows))
print("YAML file read successfully!")
print(yaml_config)

In [ ]:
# Load ALL YAML files dynamically

yaml_configs = {}

for layer, folder in yaml_folders.items():

    # Use fully qualified stage path
    folder = folder.replace(
        "@AZURE_SNOW_STAGE",
        f"@{stage_name}"
    )

    for file_path in yaml_files_by_layer[layer]:

        file_name = file_path.rsplit("/", 1)[-1]

        stage_file = f"{folder.rstrip('/')}/{file_name}"

        # Read complete YAML file
        rows = session.sql(f"""
            SELECT $1
            FROM {stage_file}
            (
                FILE_FORMAT =>
                'CUSTOMER_BILLING_DB.PUBLIC.CB_YAML_TEXT_FF'
            )
        """).collect()

        if not rows:
            raise Exception(f"Empty YAML file: {file_name}")

        yaml_text = "\n".join(
            str(row[0]) for row in rows if row[0] is not None
        )

        # Parse YAML
        parsed_config = yaml.safe_load(yaml_text)

        if not isinstance(parsed_config, dict):
            raise ValueError(f"Invalid YAML: {file_name}")

        # Store all YAML definitions
        yaml_configs[f"{layer}/{file_name}"] = parsed_config

        print(
            f"Loaded: {layer}/{file_name} | "
            f"Table: {parsed_config['table']['name']}"
        )

print("\nTotal YAML files loaded:", len(yaml_configs))

In [ ]:
# Validate all YAML definitions before creating tables

required_table_keys = ["name", "schema_name", "columns"]

for yaml_file, yaml_config in yaml_configs.items():

    if not isinstance(yaml_config, dict):
        raise ValueError(f"Invalid YAML structure: {yaml_file}")

    table_config = yaml_config.get("table")

    if not isinstance(table_config, dict):
        raise ValueError(f"Missing table definition: {yaml_file}")

    # Validate required table properties
    for key in required_table_keys:
        if key not in table_config:
            raise ValueError(
                f"Missing '{key}' in {yaml_file}"
            )

    # Validate schema and table names
    for key in ["name", "schema_name"]:
        value = table_config[key]

        if not isinstance(value, str) or not re.fullmatch(
            r"[A-Za-z_][A-Za-z0-9_]*", value
        ):
            raise ValueError(
                f"Invalid {key} in {yaml_file}: {value}"
            )

    columns = table_config["columns"]

    if not isinstance(columns, list) or not columns:
        raise ValueError(f"No columns defined: {yaml_file}")

    column_names = []

    for column in columns:

        if not isinstance(column, dict):
            raise ValueError(f"Invalid column definition: {yaml_file}")

        for key in ["name", "data_type"]:
            if key not in column:
                raise ValueError(
                    f"Missing column property '{key}': {yaml_file}"
                )

        column_name = column["name"]

        if not isinstance(column_name, str) or not re.fullmatch(
            r"[A-Za-z_][A-Za-z0-9_]*", column_name
        ):
            raise ValueError(
                f"Invalid column name in {yaml_file}: {column_name}"
            )

        if column_name.lower() in [n.lower() for n in column_names]:
            raise ValueError(
                f"Duplicate column {column_name} in {yaml_file}"
            )

        column_names.append(column_name)

        if not isinstance(column["data_type"], str):
            raise ValueError(
                f"Invalid data type in {yaml_file}: {column}"
            )

        if "nullable" in column and not isinstance(
            column["nullable"], bool
        ):
            raise ValueError(
                f"Invalid nullable setting in {yaml_file}: {column}"
            )

print("All YAML validations passed!")
print("Total YAML files validated:", len(yaml_configs))

In [ ]:
# Step 9: Create schemas and tables dynamically

database_name = config["database"]

created_tables = []

for yaml_file, yaml_config in yaml_configs.items():

    table_config = yaml_config["table"]

    schema_name = table_config["schema_name"].upper()
    table_name = table_config["name"].upper()

    columns = table_config["columns"]

    # Create schema
    session.sql(f"""
        CREATE SCHEMA IF NOT EXISTS
        {database_name}.{schema_name}
    """).collect()

    # Generate column definitions
    column_definitions = []

    for column in columns:

        column_name = column["name"].upper()
        data_type = column["data_type"].strip().upper()

        nullable = column.get("nullable", True)

        null_rule = "" if nullable else " NOT NULL"

        column_definitions.append(
            f"{column_name} {data_type}{null_rule}"
        )

    columns_sql = ",\n".join(column_definitions)

    # Generate CREATE TABLE statement
    create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS
        {database_name}.{schema_name}.{table_name}
        (
            {columns_sql}
        )
    """

    # Execute DDL
    session.sql(create_table_sql).collect()

    full_table_name = (
        f"{database_name}.{schema_name}.{table_name}"
    )

    created_tables.append(full_table_name)

    print(f"Created/Verified: {full_table_name}")

print("\n====================================")
print("DDL EXECUTION COMPLETED")
print("Total tables created/verified:", len(created_tables))
print("====================================")

In [ ]:
# # Verify all tables in Snowflake

# verification = session.sql("""
#     SELECT
#         TABLE_SCHEMA,
#         TABLE_NAME,
#         TABLE_TYPE
#     FROM CUSTOMER_BILLING_DB.INFORMATION_SCHEMA.TABLES
#     WHERE TABLE_SCHEMA IN (
#         'CONTROL',
#         'BRONZE',
#         'SILVER',
#         'GOLD'
#     )
#     AND TABLE_TYPE = 'BASE TABLE'
#     ORDER BY TABLE_SCHEMA, TABLE_NAME
# """).collect()

# for row in verification:
#     print(f"{row['TABLE_SCHEMA']}.{row['TABLE_NAME']}")

# print("\nTotal tables found:", len(verification))

# if len(verification) != 14:
#     raise Exception("Expected 14 tables. Please investigate.")

# print("All 14 tables verified successfully!")

In [ ]:
# # TEMPORARY CLEANUP — run only once

# database_name = config["database"]

# schemas_to_drop = [
#     "CONTROL",
#     "BRONZE",
#     "SILVER",
#     "GOLD"
# ]

# for schema_name in schemas_to_drop:

#     session.sql(f"""
#         DROP SCHEMA IF EXISTS
#         {database_name}.{schema_name}
#         CASCADE
#     """).collect()

#     print(f"Dropped schema: {schema_name}")

# print("All four schemas dropped successfully.")

In [ ]:
# result = session.sql("""
#     SELECT SCHEMA_NAME
#     FROM CUSTOMER_BILLING_DB.INFORMATION_SCHEMA.SCHEMATA
#     WHERE SCHEMA_NAME IN (
#         'CONTROL',
#         'BRONZE',
#         'SILVER',
#         'GOLD'
#     )
# """).collect()

# print("Remaining target schemas:", len(result))

# if len(result) == 0:
#     print("Cleanup successful!")
# else:
#     print("Some schemas still exist.")